<a href="https://colab.research.google.com/github/JianfengMI/MLprojects/blob/main/EP_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import math
from tqdm.auto import tqdm
import requests
import matplotlib.pyplot as plt

In [ ]:
# set parameters
START_DATE = (datetime.today() - timedelta(days=365*20)).strftime('%Y-%m-%d') # 5y of history // change to 20 years
END_DATE = datetime.today().strftime('%Y-%m-%d')

In [ ]:
# download S&P500 tickers and their prices
def get_sp500_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors

    # pandas can directly parse tables from HTML content
    tables = pd.read_html(response.text)

    df = None
    # Iterate through found tables to identify the S%P 500 constituents table
    for df_candidate in tables:
        # Check for columns commonly present in the S%P 500 constituents table
        # like 'Symbol', 'Ticker', or 'symbol' (case-insensitive)
        # Convert column names to string before comparison
        cols = [c for c in df_candidate.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()]
        if cols:
            df = df_candidate
            break # Found the table

    if df is None:
        raise ValueError("Could not find S%26P 500 constituents table on the Wikipedia page.")

    # Ensure 'Symbol' or 'Ticker' column is correctly identified
    # Convert column names to string before comparison
    col_name = [c for c in df.columns if "Symbol" in str(c) or "Ticker" in str(c) or "symbol" in str(c).lower()][0]

    # Clean up ticker symbols (e.g., BRK.B -> BRK-B)
    tickers = df[col_name].astype(str).str.replace(".", "-", regex=False).tolist()
    return tickers

def download_stock_prices(tickers_to_download, start, end, batch_size=80):
    """
    Download full OHLCV price data in batches using yfinance.
    Returns a MultiIndex DataFrame:
        columns = (ticker, price_field)
        index   = dates
    """
    all_data = pd.DataFrame()
    tickers_list = list(tickers_to_download)

    for i in tqdm(range(0, len(tickers_list), batch_size), desc="Downloading prices"):
        batch = tickers_list[i:i+batch_size]

        try:
            data = yf.download(
                batch, start=start, end=end,
                progress=False, group_by='ticker', auto_adjust=True
            )

            if data.empty:
                print(f"Skipping batch {batch}: No data downloaded.")
                continue

            # If single ticker, YF returns normal DataFrame → convert it to MultiIndex
            if not isinstance(data.columns, pd.MultiIndex):
                t = batch[0]
                data.columns = pd.MultiIndex.from_product([[t], data.columns])

            # Merge batch with accumulated data
            if all_data.empty:
                all_data = data
            else:
                all_data = pd.concat([all_data, data], axis=1)

        except Exception as e:
            print(f"Batch download error for {batch}: {e}")

    # Ensure sorted index and columns
    if not all_data.empty:
        all_data = all_data.sort_index()
        all_data = all_data.sort_index(axis=1)

    return all_data

In [ ]:
# Define MIN_PRICE_HISTORY_DAYS before it's used
MIN_PRICE_HISTORY_DAYS = 252 # Approximately 1 year of trading days

# Fetch tickers and prices
sp500_tickers = get_sp500_tickers()
universe = [t for t in sp500_tickers]  # copy

prices = download_stock_prices(universe, start=START_DATE, end=END_DATE)
# Remove tickers with insufficient history
valid = [t for t in sp500_tickers if prices.get(t, pd.Series()).dropna().shape[0] >= MIN_PRICE_HISTORY_DAYS]
print(f"{len(valid)} tickers with >= {MIN_PRICE_HISTORY_DAYS} days of data.")

In [ ]:
data = prices.copy()

In [ ]:
# mount on google drive and save the data
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# save the prices data
prices.to_csv('/content/drive/MyDrive/prices_20y.csv')

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/prices_20y.csv', header=[0,1], index_col=0)
data.index = pd.to_datetime(data.index)

In [ ]:
# prepare S&P 500 index data, don't keet ticker's name

spx = yf.download("^GSPC", start=START_DATE, end=END_DATE)

In [ ]:
spx.columns = spx.columns.droplevel(1)

In [ ]:
spx

In [ ]:
def label_regime(spy):

    close = spy["Close"]
    ma200 = close.rolling(200).mean()

    ret3 = close.pct_change(63)

    regime = []

    for i in range(len(close)):

        if close.iloc[i] > ma200.iloc[i] and ret3.iloc[i] > 0:
            regime.append(1)   # bull
        elif close.iloc[i] < ma200.iloc[i]:
            regime.append(-1)  # bear
        else:
            regime.append(0)   # neutral

    spy["regime"] = regime

    return spy

In [ ]:
spx = label_regime(spx)

In [ ]:
# combine data with spx['regime']
# First, align the spx['regime'] Series to the 'data' DataFrame's index.
# Using 'ffill' will propagate the last valid observation forward to fill any missing dates.
aligned_regime = spx['regime'].reindex(data.index, method='ffill')

# Iterate through each unique ticker and add the aligned 'regime' series
# as a new sub-column under that ticker in the MultiIndex DataFrame.
for ticker in data.columns.get_level_values(0).unique():
    data[ticker, 'regime'] = aligned_regime

# Institutional level EP research system

In [ ]:
# MA agreement
def ma_score(price):
    close_prices = price["Close"]

    # Ensure there's enough data for MA200 calculation
    if len(close_prices) < 80:
        return 0

    # Calculate rolling averages for the entire series
    ma10 = close_prices.rolling(10).mean()
    ma20 = close_prices.rolling(20).mean()
    ma50 = close_prices.rolling(50).mean()

    # Get MA values at 3 months ago (-63 trading days)
    ma10_3m = ma10.iloc[-63]
    ma20_3m = ma20.iloc[-63]
    ma50_3m = ma50.iloc[-63]

    # Get MA values at 6 months ago (-126 trading days)
    ma10_6m = ma10.iloc[-126]
    ma50_6m = ma50.iloc[-126]
    ma20_6m = ma20.iloc[-126]

    # Check the conditions for MA agreement
    condition_3m = (ma10_3m > ma20_3m) and (ma20_3m > ma50_3m)
    condition_6m = (ma10_6m > ma20_6m) and (ma20_6m > ma50_6m)

    if condition_3m or condition_6m:
        score = 1
    else:
        score = 0

    return score

# True base detection after trend
def detect_base_with_context(price):

    # --- Step 1: Identify prior run-up ---
    close = price["Close"]

    if len(close) < 150:
        return False, None, None

    # Use last 3–6 months to find peak
    lookback = close.iloc[-126:]  # 6 months

    peak_idx = lookback.idxmax()
    peak_price = lookback.max()

    # Ensure peak is not too recent (otherwise no base yet)
    if peak_idx == close.index[-1]:
        return False, None, None

    # --- Step 2: Define base AFTER peak ---
    base = price.loc[peak_idx:]

    if len(base) < 15:
        return False, None, None

    base_high = base["High"].max()
    base_low = base["Low"].min()

    # --- Step 3: Pullback constraint (<= 25%) ---
    drawdown = (peak_price - base_low) / peak_price

    if drawdown > 0.25:
        return False, base_high, base_low

    # --- Step 4: Compression (tight + quiet) ---
    base_range = (base_high - base_low) / base_high
    tightness = base["Close"].std() / base["Close"].mean()

    if base_range < 0.25 and tightness < 0.08:
        return True, base_high, base_low

    return False, base_high, base_low

# earnings event detection
def detect_event(price):

    gap = (
        price["Open"].iloc[-1] -
        price["Close"].iloc[-2]
    ) / price["Close"].iloc[-2]

    vol_ratio = (
        price["Volume"].iloc[-1] /
        price["Volume"].iloc[-20:].mean()
    )

    if abs(gap) > 0.03 and vol_ratio > 1.5:
        return True, gap, vol_ratio # Return all 3 values when True

    return False, gap, vol_ratio # Return all 3 values even when False

def max_runup_valid(series):
    series = series.dropna()
    if len(series) < 20:
        return 0

    cum_min = series.cummin()
    runup = series / cum_min - 1
    return runup.max()

def score_ep(price):

    trend = price["Close"].pct_change(126).iloc[-1]

    volume = price["Volume"].iloc[-1] / price["Volume"].iloc[-20:].mean()

    volatility = price["Close"].pct_change().std()

    score = (
        trend * 40 +
        volume * 30 +
        (1 / volatility) * 30
    )

    return score

# Advanced EP scanner
def advanced_ep_scan(df,lookback_days): # change to whole dataset lenghth

    signals = []

    # Get the actual unique tickers present in the first level of the MultiIndex
    actual_tickers = df.columns.get_level_values(0).unique()

    for ticker in actual_tickers:

        price = df[ticker].dropna()

        if len(price) < 150:
            continue

        # check last N days
        for i in range(lookback_days, 0, -1):
             hist = price.iloc[:-i]
             ma10 = hist["Close"].rolling(10).mean()
             ma20 = hist["Close"].rolling(20).mean()
             ma50 = hist["Close"].rolling(50).mean()

             alignment = (ma10 > ma20) & (ma20 > ma50)

             price_valid = hist["Close"].copy()
             price_valid[~alignment] = np.nan

             trend_3m = max_runup_valid(price_valid.iloc[-63:])
             trend_6m = max_runup_valid(price_valid.iloc[-126:])

             if trend_3m < 0.30 and trend_6m < 0.30:
                continue


             # Base detection WITH trend context
             base_ok, base_high, base_low = detect_base_with_context(hist)

             if not base_ok:
                continue

            # Event detection
             event_ok, gap, vol_ratio = detect_event(hist)

             if not event_ok:
                continue

             entry_price = hist["Close"].iloc[-1]
             stop_price = hist["Low"].iloc[-1]
             regime = hist["regime"].iloc[-1]

             signals.append({
                "Ticker": ticker,
                "Signal_Date": hist.index[-1],
                "Entry": entry_price,
                "Stop": stop_price,
                "Gap_%": round(gap * 100, 2),
                "Volume_Ratio": round(vol_ratio, 2),
                "Trend_3M_%": round(trend_3m * 100, 1),
                "Trend_6M_%": round(trend_6m * 100, 1),
                "BaseHigh": base_high,
                "BaseLow": base_low,
                "regime": regime,
                "Score": score_ep(hist)
            })

    signals = pd.DataFrame(signals)

    if len(signals) == 0:
        # print("No EP events in the last", lookback_days, "days")
        return signals

    signals = signals.sort_values("Signal_Date", ascending=False)

    signals['Score'] = [score_ep(data[ticker]) for ticker in signals['Ticker']]

    return signals

In [ ]:
signals = advanced_ep_scan(data, len(data)) # whole history

In [ ]:
# save signals on google drive
signals.to_csv('/content/drive/MyDrive/signals.csv')

In [ ]:
# download signals data from google drive
signals = pd.read_csv('/content/drive/MyDrive/signals.csv', index_col=0)

In [ ]:
signals

In [ ]:
signals.sort_values('Score', ascending=False)

In [ ]:
signals.sort_values('Signal_Date', ascending=False).head(8)

In [ ]:
def compute_profit_score(price, entry_date, entry_price, stop_price, max_holding_days=40):

    # --- Step 0: Start AFTER entry day ---
    trade = price.loc[entry_date:].iloc[1:].copy()

    if len(trade) == 0:
        return 0, 0 # Return two values, even if no trade data

    # --- Step 1: Apply holding cap ---
    trade = trade.iloc[:max_holding_days]

    # --- Step 2: Stop loss check (only first 3 days) ---
    early_period = trade.iloc[:3]

    if (early_period["Low"] <= stop_price).any():
        return 0, 0 # Return two values for profit and score when stop loss is hit

    # --- Step 3: Find max price and its date ---
    max_price = trade["High"].max()
    max_idx = trade["High"].idxmax()

    # --- Step 4: Compute days to peak ---
    days_to_peak = trade.index.get_loc(max_idx) + 1

    # --- Step 5: Profit ---
    profit = (max_price - entry_price) / entry_price

    # --- Step 6: Score ---
    score = profit / days_to_peak

    return profit, score

In [ ]:
results = []

for _, row in signals.iterrows():

    ticker = row["Ticker"]
    entry_date = row["Signal_Date"]
    entry_price = row["Entry"]
    stop_price = row["Stop"]

    price = data[ticker].dropna()

    profit, score = compute_profit_score(
        price,
        entry_date,
        entry_price,
        stop_price,
        max_holding_days=40
    )

    results.append({
        **row,
        "Profit_Score": score,
        "Profit": profit
    })

results = pd.DataFrame(results)

In [ ]:
results.sort_values('Profit_Score', ascending=False).head()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_trade_full(
    df,
    ticker,
    start_date=None,
    end_date=None,
    entry_date=None,
    entry_price=None,
    stop_price=None
):

    price = df[ticker].dropna().copy()

    # --- Apply window ---
    if start_date is not None:
        price = price.loc[start_date:]

    if end_date is not None:
        price = price.loc[:end_date]

    if len(price) == 0:
        print("No data in selected window.")
        return

    # --- Moving averages ---
    price["MA10"] = price["Close"].rolling(10).mean()
    price["MA20"] = price["Close"].rolling(20).mean()
    price["MA50"] = price["Close"].rolling(50).mean()

    plt.figure(figsize=(14, 7))

    plt.plot(price.index, price["Close"], label="Close", linewidth=2)
    plt.plot(price.index, price["MA10"], label="MA10")
    plt.plot(price.index, price["MA20"], label="MA20")
    plt.plot(price.index, price["MA50"], label="MA50")

    # =========================
    # 🔺 ENTRY MARKER (robust)
    # =========================
    if entry_date is not None and entry_price is not None:

        entry_date = pd.to_datetime(entry_date)

        # find nearest index
        nearest_idx = price.index.get_indexer([entry_date], method="nearest")

        if nearest_idx[0] != -1:
            entry_idx = price.index[nearest_idx[0]]

            plt.scatter(
                entry_idx,
                entry_price,
                marker="^",
                s=350,
                zorder=5,
                label="Entry"
            )

            # =========================
            # 🔻 EXIT MARKER
            # =========================
            future = price.loc[entry_idx:]
            if len(future) > 0:
                exit_idx = future["High"].idxmax()
                exit_price = future["High"].max()

                plt.scatter(
                    exit_idx,
                    exit_price,
                    marker="v",
                    s=350,
                    zorder=5,
                    label="Exit (Max)"
                )

    # =========================
    # ❌ STOP MARKER
    # =========================
    if stop_price is not None:

        stopped = price["Low"] <= stop_price

        if stopped.any():
            stop_idx = stopped.idxmax()

            plt.scatter(
                stop_idx,
                stop_price,
                marker="x",
                s=350,
                zorder=5,
                label="Stop Hit"
            )

    # --- Optional guide lines ---
    if entry_price is not None:
        plt.axhline(entry_price, linestyle="--", alpha=0.3)

    if stop_price is not None:
        plt.axhline(stop_price, linestyle="--", alpha=0.3)

        plt.title(f"{ticker} Trade View")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.grid()

    plt.show()

In [ ]:
results.sort_values('Profit', ascending=False).head(50)

In [ ]:
plot_trade_full(
    data,
    ticker="AMD",
    start_date="2022-11-01",
    end_date="2023-07-20"
)

In [ ]:
plot_trade_full(
    data,
    ticker="DELL",
    start_date="2023-02-01",
    end_date="2023-11-20"
)

# Step 1 is done. found signals and calculated score and Profit_Score (signals + Profit + Profit_Score)

## use machine learning to train results

In [ ]:
results.columns

In [ ]:
# plot the Profit_Score and Profit
import matplotlib.pyplot as plt
import seaborn as sns
results['Profit'].plot(kind='hist', bins=20, title='Profit Distribution')
plt.xlabel('Profit')
plt.ylabel('Frequency')
plt.show()

In [ ]:
results['Profit_Score'].plot(kind='hist', bins=20, title='Profit_Score Distribution')
plt.xlabel('Profit_Score')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# download fresh stock price and find signals
fresh_signals = signals.sort_values('Signal_Date', ascending=False).head(10)
fresh_signals

In [ ]:
# use the model in practice (next-day workflow)
# Load new stocks (today’s screener output)
new_df = fresh_signals.copy()

X_new = scaler.transform(new_df[features])          # same scaler!
new_df['predicted_profit_score'] = final_model.predict(X_new)

# Pick the best ones
top_picks = new_df.nlargest(5, 'predicted_profit_score')   # or top 10, top 5 %
# print(top_picks[['ticker', 'predicted_profit_score', 'gap_%', ...]])

In [ ]:
top_picks

In [ ]:
signals_results = results.copy()

# Step 2. in this step, we will change Profit_Score to categorical values and use classify model to train and predict the 0, loss or 1, gain.

## change results column "profit_Score" to categorical. 0 and 1 (0 means loss and 1 means gain)

In [ ]:
# change data results column Profit_Score to categoical value
results_cat = results.copy()
results_cat['gain'] = results_cat['Profit_Score'].apply(lambda x: 0 if x == 0 else 1)

In [ ]:
results_cat['gain'].unique()

In [ ]:
results_cat['gain'].value_counts(normalize=True)

In [ ]:
len(results_cat)

## use machine learning to train results (Logistic Regression model)

In [ ]:
# change to Logistic Regression model
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# 1. Load data
df = results_cat.copy()

df["BaseDepth"] = (df["BaseHigh"] - df["BaseLow"]) / df["BaseHigh"]
df["Gap_Strength"] = df["Gap_%"] * df["Volume_Ratio"]
df["BasePosition"] = df["Entry"] / df["BaseHigh"]
df["Vol_Expansion"] = df["Volume_Ratio"] * df["Gap_%"]
df["Trend_Regime"] = df["Trend_3M_%"] * df["regime"]
df["Volume_Regime"] = df["Volume_Ratio"] * df["regime"]

# 2. Features / Target
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline # Added this import

numeric_features = [
    'Gap_%', 'Volume_Ratio',
    'Trend_3M_%', 'Trend_6M_%',
    'BaseHigh', 'BaseLow',
    'BaseDepth', 'Gap_Strength',
    'BasePosition',
    'Vol_Expansion',
    'Score',
    'Trend_Regime',
    'Volume_Regime'
]

categorical_features = ['regime']

# 3. Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

# X = df[numeric_features + categorical_features]
# y = df['gain']

# X_processed = preprocessor.fit_transform(X)

# # 4. Train/Test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X_processed, y, test_size=0.3, random_state=42, shuffle=True
# )

# 5. Model (LightGBM)
# lgb_model = lgb.LGBMClassifier(
#     class_weight='balanced',
#     n_estimators=500,
#     learning_rate=0.03,
#     num_leaves=31,
#     min_data_in_leaf=20,
#     feature_fraction=0.8,
#     bagging_fraction=0.8,
#     bagging_freq=1,
#     random_state=42
# )

# lgb_model.fit(X_train, y_train)

from sklearn.linear_model import LogisticRegression
pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(
        class_weight='balanced',
        penalty='l1',
        solver='liblinear',
        max_iter=1000,
        C=0.5,
        random_state=42
    ))
])

X = df[numeric_features + categorical_features]
y = df['gain']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, shuffle=True
)

pipeline.fit(X, y)

# # Logistic Regression model
# log_model = LogisticRegression(
#     class_weight='balanced', # same idea as LightGBM
#     penalty='l1',
#     solver='liblinear',
#     max_iter=1000,
#     C=0.5,                     # regularization strength
#     # solver='lbfgs',
#     random_state=42
# )

# # Fit the Logistic Regression model
# log_model.fit(X_train, y_train)

# 6. Evaluation
# y_pred_proba = log_model.predict_proba(X_test)[:, 1]
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

# Better threshold than 0.5 (important)
threshold = 0.6
y_pred = (y_pred_proba >= threshold).astype(int)

print("=== Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.3f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 7. Feature importance
# Get feature names after preprocessing
feature_names_out = preprocessor.get_feature_names_out()
importances = pd.Series(pipeline['model'].coef_[0], index=feature_names_out).sort_values(ascending=False)

print("\nFeature Importance:\n", importances)

In [ ]:
# calculate the AUC-ROC
from sklearn.metrics import roc_auc_score
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.3f}")


In [ ]:
# save the trained log_model for later use
import joblib

joblib.dump(pipeline, '/content/drive/MyDrive/log_pipeline.pkl')

In [ ]:
# download model

from google.colab import files

files.download('/content/drive/MyDrive/log_pipeline.pkl')

In [ ]:
# use the model for later
import joblib

pipeline = joblib.load('log_pipeline.pkl')

pred = pipeline.predict_proba(new_df)[:, 1]

In [ ]:
#### fine tune the model
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

param_grid = {
    'model__C': [0.01, 0.1, 0.5, 1, 2, 5],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear'],   # supports both l1 and l2
    'model__class_weight': [None, 'balanced']
}

from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best ROC-AUC:", grid.best_score_)

best_model = grid.best_estimator_

In [ ]:
# try Random forest model
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,           # important: keep shallow (small data)
    min_samples_leaf=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict_proba(X_test)[:, 1]

threshold = 0.6
y_pred = (rf_pred >= threshold).astype(int)

print("=== Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.3f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
# Apply a threshold to ensemble_pred to convert it into binary predictions
ensemble_pred_binary = (ensemble_pred >= threshold).astype(int)

print(f"Accuracy:  {accuracy_score(y_test, ensemble_pred_binary):.3f}")
print(f"Precision: {precision_score(y_test, ensemble_pred_binary):.3f}")
print(f"Recall:    {recall_score(y_test, ensemble_pred_binary):.3f}")
print(f"F1-Score:  {f1_score(y_test, ensemble_pred_binary):.3f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, ensemble_pred_binary))

In [ ]:
from sklearn.pipeline import Pipeline

log_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        class_weight='balanced',
        penalty='l1',
        solver='liblinear',
        C=0.5
    ))
])

In [ ]:
## try stacking models
from sklearn.ensemble import StackingClassifier

stack_model = StackingClassifier(
    estimators=[
        ('lr', log_pipeline),
        ('rf', RandomForestClassifier(
            n_estimators=300,
            max_depth=5,
            min_samples_leaf=10,
            random_state=42
        ))
    ],
    final_estimator=LogisticRegression(),
    passthrough=True
)

In [ ]:
stack_model.fit(X_train, y_train)

stack_pred = stack_model.predict_proba(X_test)[:, 1]

In [ ]:
stack_pred_binary = (stack_pred >= threshold).astype(int)

print(f"Accuracy:  {accuracy_score(y_test, stack_pred_binary):.3f}")
print(f"Precision: {precision_score(y_test, stack_pred_binary):.3f}")
print(f"Recall:    {recall_score(y_test, stack_pred_binary):.3f}")
print(f"F1-Score:  {f1_score(y_test, stack_pred_binary):.3f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, stack_pred_binary))

In [ ]:
# save the lgb_model on google drive
lgb_model.booster_.save_model('/content/drive/MyDrive/lgb_model.txt')

## categorical seems work. now build a function to generate features for model training (1. find signals, 2, build features)

In [ ]:
##################
# ignore now
##################

def build_ep_vcp_features(price):

    close = price["Close"]
    high = price["High"]
    low = price["Low"]
    vol = price["Volume"]

    features = {}

    # =========================
    # 1. TREND STRENGTH
    # =========================
    features["Trend_3M"] = close.pct_change(63).iloc[-1]
    features["Trend_6M"] = close.pct_change(126).iloc[-1]

    features["Trend_Accel"] = features["Trend_3M"] - features["Trend_6M"]

    # =========================
    # 2. BASE STRUCTURE
    # =========================
    base = price.iloc[-40:]

    base_high = base["High"].max()
    base_low = base["Low"].min()

    features["BaseDepth"] = (base_high - base_low) / base_high
    features["BaseTightness"] = base["Close"].std() / base["Close"].mean()

    # Position in base (how close to breakout)
    features["BreakoutProximity"] = close.iloc[-1] / base_high

    # =========================
    # 3. VOLATILITY CONTRACTION (VCP core)
    # =========================
    ranges = (high - low) / close

    r1 = ranges.iloc[-60:-40].mean()
    r2 = ranges.iloc[-40:-20].mean()
    r3 = ranges.iloc[-20:].mean()

    features["Vol_Contract_1"] = r1 - r2
    features["Vol_Contract_2"] = r2 - r3

    features["Vol_Contract_Ratio"] = r3 / r1 if r1 > 0 else 0

    # =========================
    # 4. HIGHER LOWS (ACCUMULATION)
    # =========================
    lows = low.rolling(5).min()

    features["HigherLow"] = 1 if lows.iloc[-1] > lows.iloc[-20] else 0

    # =========================
    # 5. VOLUME DRY-UP
    # =========================
    vol_short = vol.iloc[-5:].mean()
    vol_long = vol.iloc[-30:].mean()

    features["Vol_Dry"] = vol_short / vol_long

    # =========================
    # 6. GAP + EVENT QUALITY
    # =========================
    gap = (price["Open"].iloc[-1] - price["Close"].iloc[-2]) / price["Close"].iloc[-2]
    vol_ratio = vol.iloc[-1] / vol.iloc[-20:].mean()

    features["Gap"] = gap
    features["Vol_Ratio"] = vol_ratio
    features["Gap_Strength"] = gap * vol_ratio

    # =========================
    # 7. TREND QUALITY (SMOOTHNESS)
    # =========================
    returns = close.pct_change().dropna()

    features["Trend_Sharpe"] = returns.mean() / returns.std() if returns.std() > 0 else 0

    return features

In [ ]:
rows = []

for _, row in results_cat.iterrows():

    ticker = row["Ticker"]
    date = row["Signal_Date"]

    price = data[ticker].loc[:date].dropna()

    feats = build_ep_vcp_features(price)

    feats["Target"] = row["Profit_Score"]
    feats['regime'] = row['regime']

    rows.append(feats)

dataset = pd.DataFrame(rows)

In [ ]:
dataset.columns

In [ ]:
##################
# ignore now
##################
# use new dataset to train LightGBM model again

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# 1. Load data
df = dataset.copy()

# 2. Features / Target
features = [
    'Trend_3M',
    'Trend_6M',
    'Trend_Accel',
    'BaseDepth',
    'BaseTightness',
    'BreakoutProximity',
    'Vol_Contract_1',
    'Vol_Contract_2',
    'Vol_Contract_Ratio',
    'HigherLow',
    'Vol_Dry',
    'Gap',
    'Vol_Ratio',
    'Gap_Strength',
    'Trend_Sharpe',
    'regime'
]

X = df[features]
y = df['Target']

# 3. Preprocessing
X = X.fillna(X.median())

# scaler = StandardScaler()
X_scaled = X.values

# 4. Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, shuffle=True
)

# 5. Model (LightGBM)
model = lgb.LGBMClassifier(
    class_weight='balanced',
    n_estimators=300,
    learning_rate=0.05,
    um_leaves=63,        # ↑ more flexibility
    min_data_in_leaf=10,  # ↓ allow smaller splits
    min_gain_to_split=0.0,
    random_state=42
)

model.fit(X_train, y_train)

# 6. Evaluation
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Better threshold than 0.5 (important)
threshold = 0.6
y_pred = (y_pred_proba >= threshold).astype(int)

print("=== Model Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.3f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 7. Feature importance
importances = pd.Series(model.feature_importances_, index=features)\
                .sort_values(ascending=False)

print("\nFeature Importance:\n", importances)

# Step 3. Now we can predict the loss or gian tickers, next we want to use positive results table to train regression model to predict the Profit_Score (or the power of gain)

In [ ]:
gain_results = signals_results[signals_results['Profit_Score'] > 0]

In [ ]:
# get the training dataset
##################
# ignore now
##################
rows = []

for _, row in gain_results.iterrows():

    ticker = row["Ticker"]
    date = row["Signal_Date"]

    price = data[ticker].loc[:date].dropna()

    feats = build_ep_vcp_features(price)

    feats["Profit_Score"] = row["Profit_Score"]
    feats['regime'] = row['regime']

    rows.append(feats)

reg_dataset = pd.DataFrame(rows)

In [ ]:
reg_dataset.columns

In [ ]:
gain_results['Profit_Score'].plot(kind='hist', bins=20, title='Profit_Score Distribution')

In [ ]:
gain_results[gain_results['regime'] == 1]['Profit_Score'].plot(kind='hist', bins=20, title='Profit_Score Distribution')

In [ ]:
gain_results['Profit'].plot(kind='hist',bins=20, title='Profit_Score Distribution')

In [ ]:
gain_results[gain_results['regime'] == 1]['Profit'].plot(kind='hist', bins=20, title='Profit Distribution')

In [ ]:
gain_results.columns

In [ ]:
# use XGBRegressor regression model
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# 1. Load data
df = gain_results.copy()

# 2. Features / Target
features = ['Gap_%', 'Volume_Ratio',
       'Trend_3M_%', 'Trend_6M_%', 'BaseHigh', 'BaseLow', 'regime', 'Score']
X = df[features]
y = np.log1p(df["Profit"])

# 3. Preprocessing
X = X.fillna(X.median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, shuffle=True
)

# 5. Model (Regressor)
reg_model = xgb.XGBRegressor(
    objective="reg:squaredlogerror",
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

# Pass the groups specific to the training data
reg_model.fit(X_train, y_train)

# 6. Evaluation
y_pred = reg_model.predict(X_test)

print("=== Model Performance ===")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R2:  {r2_score(y_test, y_pred):.4f}")

# 7. Feature importance
importances = pd.Series(reg_model.feature_importances_, index=features)\
                .sort_values(ascending=False)

print("\nFeature Importance:\n", importances)

In [ ]:
# tune the parameters
grid = {
    'n_estimators': [100, 200,300,400],
    'learning_rate': [0.01, 0.05,0.01],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 0.9]
}
grid_search = GridSearchCV(estimator=reg_model, param_grid=grid, scoring='neg_mean_absolute_error', cv=5)
grid_search.fit(X_train, y_train)

In [ ]:
best_params = grid_search.best_params_
best_score_from_cv = grid_search.best_score_    # this is usually negative MAE/RMSE, so interpret accordingly

print("Best hyperparameters:", best_params)
print("Best CV score:", best_score_from_cv)

In [ ]:
final_reg_model = xgb.XGBRegressor(**best_params, random_state=42)
# Use groups_train if you intend to fit on X_train/y_train in this cell
# However, the final_reg_model is already fitted on the full data in cell KcvIYxSmXK0E.
final_reg_model.fit(X_train, y_train)

In [ ]:
# save final_reg_model
final_reg_model.save_model('/content/drive/MyDrive/final_reg_model.txt')

In [ ]:
# plot the ture Profit_Score and the predict
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# final_model
# X_test, y_test
# scaler (if you used it)

y_pred = final_reg_model.predict(X_test)
y_pred = np.expm1(y_pred)
y_test = np.expm1(y_test)

# Create a DataFrame for easier plotting
pred_results = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred
})

# Add a perfect prediction line
min_val = min(pred_results['actual'].min(), pred_results['predicted'].min())
max_val = max(pred_results['actual'].max(), pred_results['predicted'].max())

plt.figure(figsize=(10, 8))
sns.scatterplot(data=pred_results, x='actual', y='predicted', alpha=0.6, s=80)

# Diagonal reference line
plt.plot([min_val, max_val], [min_val, max_val],
         color='red', linestyle='--', linewidth=2, label='Perfect prediction')

plt.xlabel('Actual profit_score', fontsize=12)
plt.ylabel('Predicted profit_score', fontsize=12)
plt.title('Predicted vs Actual Profit Score', fontsize=14)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
pred_results.corr()

In [ ]:
pred_results[["predicted", "actual"]].corr()

# This section is trying to resolve the small data size problem

In [ ]:
dataset.head()

# Daily Top-N EP System:
Each day:

Scan signals

Score with model

Pick top N

Track forward performance

## Step 1. download recent stock prices ans spx history data. find recent (10 days) signals.

In [ ]:
# set parameters
START_DATE_1y = (datetime.today() - timedelta(days=365*1)).strftime('%Y-%m-%d') # 1y histroy data
END_DATE = datetime.today().strftime('%Y-%m-%d')

In [ ]:
# Define MIN_PRICE_HISTORY_DAYS before it's used
MIN_PRICE_HISTORY_DAYS_1y = 200 # Approximately 1 year of trading days

# Fetch tickers and prices
sp500_tickers = get_sp500_tickers()
universe = [t for t in sp500_tickers]  # copy

prices_1y = download_stock_prices(universe, start=START_DATE_1y, end=END_DATE)
# Remove tickers with insufficient history
valid = [t for t in sp500_tickers if prices_1y.get(t, pd.Series()).dropna().shape[0] >= MIN_PRICE_HISTORY_DAYS_1y]
print(f"{len(valid)} tickers with >= {MIN_PRICE_HISTORY_DAYS_1y} days of data.")

In [ ]:
spx_1y = yf.download("^GSPC", start=START_DATE_1y, end=END_DATE)
spx_1y.columns = spx_1y.columns.droplevel(1)
spx_1y = label_regime(spx_1y)

In [ ]:
data_1y = prices_1y.copy()
aligned_regime = spx_1y['regime'].reindex(data_1y.index, method='ffill')

for ticker in data_1y.columns.get_level_values(0).unique():
    data_1y[ticker, 'regime'] = aligned_regime

In [ ]:
signals_1y = advanced_ep_scan(data_1y, 40)  # only show recent signals

In [ ]:
signals_1y

## Step 2. get the features and scale them. use LightGBM model to predict whether they are profitable.

In [ ]:
df = signals_1y.copy()

# 1. Feature extract
df["BaseDepth"] = (df["BaseHigh"] - df["BaseLow"]) / df["BaseHigh"]
df["Gap_Strength"] = df["Gap_%"] * df["Volume_Ratio"]
df["BasePosition"] = df["Entry"] / df["BaseHigh"]
df["Vol_Expansion"] = df["Volume_Ratio"] * df["Gap_%"]
df["Trend_Regime"] = df["Trend_3M_%"] * df["regime"]
df["Volume_Regime"] = df["Volume_Ratio"] * df["regime"]

# 2. Features / Target
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = [
    'Gap_%', 'Volume_Ratio',
    'Trend_3M_%', 'Trend_6M_%',
    'BaseHigh', 'BaseLow',
    'BaseDepth', 'Gap_Strength',
    'BasePosition',
    'Vol_Expansion',
    'Score',
    'Trend_Regime',
    'Volume_Regime'
]

categorical_features = ['regime']

# 3. Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

X = df[numeric_features + categorical_features]

X_processed = preprocessor.fit_transform(X)

In [ ]:
probabilities = log_model.predict_proba(X_processed)[:, 1]
signals_1y['gain'] = np.where(probabilities > 0.6, 1, 0)

In [ ]:
signals_1y

## Step 3. Use second regression model to predict the profit

In [ ]:
# 1. Load data
df_2 = signals_1y.copy()

# 2. Features / Target
features = ['Gap_%', 'Volume_Ratio',
       'Trend_3M_%', 'Trend_6M_%', 'BaseHigh', 'BaseLow', 'regime', 'Score']
X = df_2[features]

# 3. Preprocessing
X = X.fillna(X.median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
signals_1y['profit'] = np.expm1(final_reg_model.predict(X_scaled))

In [ ]:
signals_1y

# Stop here

## Step 2: Build Daily Top-N selector

In [ ]:
def select_top_n(signals, n=5, prob_threshold=0.6):

    # Filter high-quality signals
    filtered = signals[signals["Pred_Prob"] >= prob_threshold]

    # Rank within each day
    top_n = (
        filtered
        .sort_values(["Signal_Date", "Pred_Prob"], ascending=[True, False])
        .groupby("Signal_Date")
        .head(n)
    )

    return top_n

## Step 3: Trade simulation engine (per trade)

In [ ]:
def simulate_trade(price, entry_date, entry_price, stop_price, max_days=40):

    trade = price.loc[entry_date:].iloc[1:].copy()

    if len(trade) == 0:
        return 0

    trade = trade.iloc[:max_days]

    stopped = trade["Low"] <= stop_price

    if stopped.any():
        stop_idx = stopped.idxmax()
        exit_price = stop_price
        exit_date = stop_idx
    else:
        exit_price = trade["High"].max()
        exit_date = trade["High"].idxmax()

    return (exit_price - entry_price) / entry_price

In [ ]:
## Step 4: Portfolio backtest (core system)

In [ ]:
def backtest_top_n(df, signals, model, features, n=5, prob_threshold=0.6):

    results = []

    # Sort by date (IMPORTANT)
    signals = signals.sort_values("Signal_Date")

    print(f"Total signals input to backtest_top_n: {len(signals)}")
    print(f"Using probability threshold: {prob_threshold}")

    for date, group in signals.groupby("Signal_Date"):

        # Build feature matrix for that day
        # Ensure features match the model's training features
        X_day_raw = group[features]
        X_day = X_day_raw.fillna(0) # Handle NaNs as done during training

        # Predict probabilities
        probs = model.predict_proba(X_day)[:, 1]

        group = group.copy()
        group["Pred_Prob"] = probs

        # Select top N
        selected = group[group["Pred_Prob"] >= prob_threshold]\
                    .sort_values("Pred_Prob", ascending=False)\
                    .head(n)

        if not selected.empty:
            print(f"  On {date}, selected {len(selected)} signals.")

        # Simulate trades
        for _, row in selected.iterrows():

            ticker = row["Ticker"]
            # Ensure 'df' is the actual price data (multi-index DataFrame)
            price = df[ticker].dropna()

            entry_date = row["Signal_Date"]
            entry_price = row["Entry"]
            stop_price = row["Stop"]

            ret = simulate_trade(price, entry_date, entry_price, stop_price)

            results.append({
                "Date": entry_date,
                "Ticker": ticker,
                "Return": ret,
                "Prob": row["Pred_Prob"]
            })

    print(f"Total trades generated by backtest_top_n: {len(results)}")
    return pd.DataFrame(results)

## Step 5: Portfolio performance

In [ ]:
def evaluate_portfolio(trades):

    if len(trades) == 0:
        print("No trades")
        return

    trades = trades.copy()

    trades["Equity"] = (1 + trades["Return"]).cumprod()

    print("Total Trades:", len(trades))
    print("Win Rate:", (trades["Return"] > 0).mean())
    print("Avg Return:", trades["Return"].mean())
    print("Total Return:", trades["Equity"].iloc[-1] - 1)

    return trades

## Run the full system

In [ ]:
# Redefine features for lgb_model
features = [
    'Gap_%', 'Volume_Ratio',
    'Trend_3M_%', 'Trend_6M_%',
    'BaseHigh', 'BaseLow',
    'BaseDepth', 'Gap_Strength',
    'BasePosition',
    'Vol_Expansion',
    'regime',
    'Score'
]

trades = backtest_top_n(
    data, # Pass the original 'data' DataFrame for stock prices
    signals, # Pass the 'signals' DataFrame with Pred_Prob
    lgb_model, # Pass the trained 'lgb_model' here, not 'model' as it refers to XGBoost for AI ranking
    features, # Use the 'features' list for the lgb_model
    n=5,
    prob_threshold=0.6
)

performance = evaluate_portfolio(trades)

# stop here. Maybe need to change the base (sidewalk)

In [ ]:
def scan_stock_ep(price):

    signals = []

    close = price["Close"].values
    high = price["High"].values
    low = price["Low"].values
    openp = price["Open"].values
    volume = price["Volume"].values
    dates = price.index

    for i in range(150, len(price)-10):

        # ----- Trend -----
        trend = close[i] / close[i-126] - 1
        if trend < 0.2:
            continue

        # ----- Base -----
        base_high = high[i-40:i-5].max()
        base_low = low[i-40:i-5].min()

        base_range = (base_high - base_low) / base_high
        tightness = close[i-40:i-5].std() / close[i-40:i-5].mean()

        if base_range > 0.35 or tightness > 0.08:
            continue

        # ----- Event -----
        gap = (openp[i] - close[i-1]) / close[i-1]

        vol_ratio = volume[i] / volume[i-20:i].mean()

        close_strength = (close[i] - low[i]) / (high[i] - low[i] + 1e-6)

        if not (abs(gap) > 0.05 and vol_ratio > 1.8 and close_strength > 0.6):
            continue

        signals.append({
            "Date": dates[i],
            "Entry": close[i],
            "Stop": low[i],
            "Gap": gap,
            "VolRatio": vol_ratio
        })

    return signals

In [ ]:
def backtest_trade(price, start_index, entry, stop):

    close = price["Close"].values

    risk = entry - stop

    for j in range(start_index+1, min(start_index+40, len(price))):

        # stop loss
        if close[j] < stop:
            return -1

        # 3R target
        if close[j] > entry + 3*risk:
            return 3

    return (close[j] - entry) / risk

In [ ]:
results = []

for ticker in data.columns.levels[0]:

    price = data[ticker].dropna()

    if len(price) < 200:
        continue

    signals = scan_stock_ep(price)

    close = price["Close"].values

    for s in signals:

        idx = price.index.get_loc(s["Date"])

        r = backtest_trade(price, idx, s["Entry"], s["Stop"])

        results.append({
            "Date": s["Date"],
            "Ticker": ticker,
            "Entry": s["Entry"],
            "Stop": s["Stop"],
            "Gap": s["Gap"],
            "R": r
        })

results = pd.DataFrame(results)

In [ ]:
print("Trades:", len(results))
print("Avg R:", results["R"].mean())
print("Win Rate:", (results["R"] > 0).mean())

In [ ]:
results

In [ ]:
equity_curve = [100000]
for r in results.sort_values('Date')['R']:
    pos = equity_curve[-1] * 0.05
    equity_curve.append(equity_curve[-1] + pos * r)

pd.Series(equity_curve).plot(title="EP Portfolio Simulation")

## True AI ranking system

In [ ]:
# Feature extraction function (need function scan_stock_ep(price))
def extract_features(price):

    close = price["Close"]
    volume = price["Volume"]

    features = {}

    # momentum
    features["ret_1m"] = close.pct_change(21).iloc[-1]
    features["ret_3m"] = close.pct_change(63).iloc[-1]
    features["ret_6m"] = close.pct_change(126).iloc[-1]

    # trend
    ma50 = close.rolling(50).mean()
    ma200 = close.rolling(200).mean()

    features["dist_ma50"] = close.iloc[-1]/ma50.iloc[-1] - 1
    features["dist_ma200"] = close.iloc[-1]/ma200.iloc[-1] - 1

    # base structure
    base = price.iloc[-40:-5]

    features["base_depth"] = (
        base["High"].max() - base["Low"].min()
    ) / base["High"].max()

    features["base_volatility"] = base["Close"].std()

    # volume
    features["volume_spike"] = (
        volume.iloc[-1] / volume.rolling(20).mean().iloc[-1]
    )

    # volatility
    features["daily_volatility"] = close.pct_change().std()

    return features

In [ ]:
# labeling trades
def compute_future_R(price, signal):

    entry = signal["Entry"]
    stop = signal["Stop"]

    risk = entry - stop

    future = price.loc[signal["Date"]:]

    if len(future) < 20:
        return None

    high = future["High"].iloc[:20].max()

    R = (high - entry) / risk

    return 1 if R >= 3 else 0

In [ ]:
# building training dataset
def build_training_data(df, signals):

    rows = []

    for _, s in signals.iterrows():

        ticker = s["Ticker"]
        date = s["Date"]

        price = df[ticker].loc[:date]

        features = extract_features(price)

        label = compute_future_R(df[ticker], s)

        rows.append({**features, "label": label})

    return pd.DataFrame(rows)

In [ ]:
# training the model
from sklearn.ensemble import RandomForestClassifier

def train_model(dataset):

    X = dataset.drop("label", axis=1)
    y = dataset["label"]

    model = RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=10
    )

    model.fit(X, y)

    return model

In [ ]:
# build the training dataset
def build_dataset(df, signals):

    rows = []

    for _, s in signals.iterrows():

        ticker = s["Ticker"]
        date = s["Date"]

        price = df[ticker].loc[:date]

        if len(price) < 200:
            continue

        features = extract_features(price)

        label = compute_future_R(df[ticker], s)

        if label is None:
            continue

        rows.append({
            **features,
            "label": label,
            "year": date.year # Add the 'year' column
        })

    return pd.DataFrame(rows)

In [ ]:
# train XGBoost model
import xgboost as xgb

def train_xgboost(dataset):

    X = dataset.drop(["label", "year"], axis=1) # Drop 'year' column from features
    y = dataset["label"]

    model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X, y)

    return model

In [ ]:
# Walk-forward training (early data for training, later data for test)
def walk_forward_training(dataset, years):

    results = []

    for test_year in years:

        train = dataset[dataset["year"] < test_year]
        test = dataset[dataset["year"] == test_year]

        if len(train) < 100:
            continue

        model = train_xgboost(train)

        X_test = test.drop(["label","year"], axis=1)

        probs = model.predict_proba(X_test)[:,1]

        test["prob"] = probs

        results.append(test)

    return pd.concat(results)

In [ ]:
# AI ranking engine (rank signals using predicted probability)
def ai_rank_signals(df, signals, model):

    rows = []

    for _, s in signals.iterrows():

        ticker = s["Ticker"]
        date = s["Date"]

        price = df[ticker].loc[:date]

        # Use super_features for super_model, extract_features for the other model
        if model == super_model:
            features = super_features(price)
        else:
            features = extract_features(price)

        prob = model.predict_proba([list(features.values())])[0][1]

        rows.append({
            "Ticker": ticker,
            "AI_score": prob
        })

    ranked = pd.DataFrame(rows)

    return ranked.sort_values("AI_score", ascending=False)

In [ ]:
df = data.copy()

In [ ]:
dataset = build_dataset(df, results)

In [ ]:
dataset

In [ ]:
unique_years = sorted(dataset['year'].unique())
# Use the last 3 years for walk-forward testing
years_to_test = unique_years[-3:]

# Perform walk-forward evaluation (optional, store results if needed)
# walk_forward_eval_results = walk_forward_training(dataset, years=years_to_test)

# Train a final model on the entire dataset for current day predictions
# The train_xgboost function already handles dropping 'label' and 'year' from features
model = train_xgboost(dataset)

In [ ]:
all_today_signals = []
for ticker in data.columns.levels[0]:
    price_for_ticker = data[ticker].dropna()
    # Ensure enough data for calculations in scan_stock_ep
    if len(price_for_ticker) < 200:
        continue

    signals_for_current_ticker = scan_stock_ep(price_for_ticker)

    # Add ticker information to each signal dictionary
    for signal in signals_for_current_ticker:
        signal['Ticker'] = ticker
        all_today_signals.append(signal)

today_signals = pd.DataFrame(all_today_signals)

if today_signals.empty:
    print("No signals found for today.")
else:
    print(f"Found {len(today_signals)} signals for today:")
    print(today_signals.head()) # Print head for brevity

In [ ]:
ranked = ai_rank_signals(df, today_signals, model)

In [ ]:
ranked

In [ ]:
# Feature importance
importances = model.feature_importances_

In [ ]:
importances_df = pd.DataFrame({
    'Feature': dataset.drop(['label', 'year'], axis=1).columns,
    'Importance': importances
})

In [ ]:
importances_df

# superperformance detector

normal EP system to find 2-5R trades superperformance detector to find stocks capable of 5-10x moves

    Earnings acceleration (30% - 60% - 100%)
    Liquidity shock (3-10x average volumes)
    Tigh consolidation (Large funds accumulate shares quietly)
    Relative strength breakout (stock oupperforms the market before the price breakout)
    Float constrains (lower float stocks move faster)



In [ ]:
# Superperformance features
def super_features(price):

    close = price["Close"]
    volume = price["Volume"]

    f = {}

    # momentum
    f["ret_6m"] = close.pct_change(126).iloc[-1]
    f["ret_12m"] = close.pct_change(252).iloc[-1]

    # trend alignment
    ma50 = close.rolling(50).mean()
    ma200 = close.rolling(200).mean()

    f["trend_strength"] = ma50.iloc[-1] / ma200.iloc[-1]

    # base tightness
    base = price.iloc[-40:-5]

    f["base_depth"] = (
        base["High"].max() - base["Low"].min()
    ) / base["High"].max()

    # volatility contraction
    f["base_volatility"] = base["Close"].std()

    # volume expansion
    f["volume_spike"] = (
        volume.iloc[-1] /
        volume.rolling(20).mean().iloc[-1]
    )

    # volatility regime
    f["daily_volatility"] = close.pct_change().std()

    return f

In [ ]:
# Labeling superperformers
def label_superperformers(price, date):

    future = price.loc[date:]

    if len(future) < 500:
        return None

    future_max = future["High"].iloc[:500].max()

    start_price = price.loc[date]["Close"]

    return 1 if future_max / start_price >= 5 else 0

In [ ]:
# Train the Superperformance Model
import xgboost as xgb

def train_super_model(dataset):

    X = dataset.drop(["label", "year"], axis=1) # Drop 'year' column from features
    y = dataset["label"]

    model = xgb.XGBClassifier(
        n_estimators=600,
        max_depth=6,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8
    )

    model.fit(X, y)

    return model

In [ ]:
# Detect superperformance candidates
def detect_superperformers(df, model):

    rows = []

    for ticker in df.columns.levels[0]:

        price = df[ticker].dropna()

        if len(price) < 250:
            continue

        f = super_features(price)

        prob = model.predict_proba([list(f.values())])[0][1]

        rows.append({
            "Ticker": ticker,
            "SuperScore": prob
        })

    ranked = pd.DataFrame(rows)

    return ranked.sort_values("SuperScore", ascending=False)

In [ ]:
# build the training dataset
def build_super_dataset(df, signals):

    rows = []

    for _, s in signals.iterrows():

        ticker = s["Ticker"]
        date = s["Date"]

        price = df[ticker].loc[:date]

        if len(price) < 200:
            continue

        features = super_features(price)

        label = label_superperformers(df[ticker], s['Date']) # Pass only the date from the Series 's'

        if label is None:
            continue

        rows.append({
            **features,
            "label": label,
            "year": date.year # Add the 'year' column
        })

    return pd.DataFrame(rows)

In [ ]:
super_dataset = build_super_dataset(df, results)

In [ ]:
unique_years = sorted(super_dataset['year'].unique())
# Use the last 3 years for walk-forward testing
years_to_test = unique_years[-3:]

# Perform walk-forward evaluation (optional, store results if needed)
# walk_forward_eval_results = walk_forward_training(dataset, years=years_to_test)

# Train a final model on the entire dataset for current day predictions
# The train_super_model function already handles dropping 'label' and 'year' from features
super_model = train_super_model(super_dataset)

In [ ]:
all_today_signals = []
for ticker in data.columns.levels[0]:
    price_for_ticker = data[ticker].dropna()
    # Ensure enough data for calculations in scan_stock_ep
    if len(price_for_ticker) < 200:
        continue

    signals_for_current_ticker = scan_stock_ep(price_for_ticker)

    # Add ticker information to each signal dictionary
    for signal in signals_for_current_ticker:
        signal['Ticker'] = ticker
        all_today_signals.append(signal)

today_signals = pd.DataFrame(all_today_signals)

if today_signals.empty:
    print("No signals found for today.")
else:
    print(f"Found {len(today_signals)} signals for today:")
    print(today_signals.head()) # Print head for brevity

In [ ]:
super_ranked = ai_rank_signals(df, today_signals, super_model)

In [ ]:
super_ranked.head(10)

## Adding a Market Regime Model

In [ ]:
def download_market():
  spy = yf.download(
      tickers="SPY",
      period="5y")
  return spy

In [ ]:
# Market features (use indicators describe the market environment)
def market_features_df(spy):
    close = spy["Close"]

    df = spy.copy()

    df["ret_1m"] = close.pct_change(21)
    df["ret_3m"] = close.pct_change(63)

    ma50 = close.rolling(50).mean()
    ma200 = close.rolling(200).mean()

    df["trend"] = close / ma200
    df["ma_cross"] = ma50 / ma200
    df["volatility"] = close.pct_change().rolling(20).std()

    return df

In [ ]:
# get the spy features
spy = download_market()

In [ ]:
spy.columns = spy.columns.droplevel(1)

In [ ]:
spy

In [ ]:
spy_feat_df = market_features_df(spy)

In [ ]:
spy_feat_df

In [ ]:
# Regime labeling
#####################################################
# Bull  → SPY above 200MA and positive 3-month return
# Bear  → SPY below 200MA
# Neutral → everything else
#####################################################
def label_regime(spy):

    close = spy["Close"]
    ma200 = close.rolling(200).mean()

    ret3 = close.pct_change(63)

    regime = []

    for i in range(len(close)):

        if close.iloc[i] > ma200.iloc[i] and ret3.iloc[i] > 0:
            regime.append(1)   # bull
        elif close.iloc[i] < ma200.iloc[i]:
            regime.append(-1)  # bear
        else:
            regime.append(0)   # neutral

    spy["regime"] = regime

    return spy

In [ ]:
spy_regime  = label_regime(spy_feat_df)

In [ ]:
spy_regime['regime'].unique()

In [ ]:
# Train Regime Model
from sklearn.ensemble import RandomForestClassifier

def train_regime_model(dataset):

    X = dataset.drop("regime", axis=1)
    y = dataset["regime"]

    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=5
    )

    model.fit(X, y)

    return model

In [ ]:
# Predict current regime (1, bull; 0, neutral; -1, bear)
def detect_market_regime(spy, model):

    f_df = market_features_df(spy)
    # Select the last row of features, drop any NaNs that might result from rolling windows
    # and reshape it to a 2D array for the model.predict method.
    current_features = f_df.iloc[-1].dropna().values.reshape(1, -1)

    prob = model.predict(current_features)[0]

    return prob

In [ ]:
# Position size adjustment
def position_size(base_size, regime):

    if regime == 1:
        return base_size

    if regime == 0:
        return base_size * 0.5

    if regime == -1:
        return 0

In [ ]:
# train regime model
regime_model = train_regime_model(spy_regime)

In [ ]:
# detect regime
regime_prob = detect_market_regime(spy, regime_model)

In [ ]:
regime_prob

## Liquidity & Institutional Accumulation Detector

The detector looks for three main signals:

Volume accumulation

Price support during pullbacks

Volatility contraction with rising demand

In [ ]:
# Accumulation Features
def accumulation_features(price):

    close = price["Close"]
    volume = price["Volume"]
    high = price["High"]
    low = price["Low"]

    features = {}

    # 1. Volume expansion vs average
    features["volume_ratio"] = volume.iloc[-1] / volume.rolling(50).mean().iloc[-1]

    # 2. Up-volume dominance
    up_volume = volume[close > close.shift()]
    down_volume = volume[close < close.shift()]

    features["volume_balance"] = up_volume.sum() / (down_volume.sum() + 1)

    # 3. Price holding near highs
    range_pos = (close.iloc[-1] - low.iloc[-20:].min()) / (high.iloc[-20:].max() - low.iloc[-20:].min())
    features["range_position"] = range_pos

    # 4. Volatility contraction
    vol1 = close.pct_change().rolling(10).std().iloc[-1]
    vol2 = close.pct_change().rolling(30).std().iloc[-1]

    features["volatility_contraction"] = vol2 - vol1

    # 5. Accumulation trend
    features["price_trend"] = close.iloc[-1] / close.rolling(60).mean().iloc[-1]

    return features

In [ ]:
# Accumulation Score
def accumulation_score(features):

    score = (
        features["volume_ratio"] * 25 +
        features["volume_balance"] * 25 +
        features["range_position"] * 20 +
        features["volatility_contraction"] * 15 +
        features["price_trend"] * 15
    )

    return score

In [ ]:
# Accumulation Scanner
import pandas as pd

def scan_accumulation(df):

    results = []

    for ticker in df.columns.levels[0]:

        price = df[ticker].dropna()

        if len(price) < 100:
            continue

        f = accumulation_features(price)

        score = accumulation_score(f)

        results.append({
            "Ticker": ticker,
            "AccumulationScore": score
        })

    ranked = pd.DataFrame(results)

    return ranked.sort_values("AccumulationScore", ascending=False)

In [ ]:
# Which stocks are quietly being accumulated right now?
ranked = scan_accumulation(df)

In [ ]:
# Pre-filter before EP scan
top_candidates = ranked.head(50)["Ticker"].tolist()

df_filtered = df[top_candidates]

In [ ]:
# The advanced_ep_scan function expects a MultiIndex DataFrame.
# df_filtered is already a MultiIndex DataFrame containing only the top_candidates.
today_signals = advanced_ep_scan(df_filtered, lookback_days=10)

if today_signals.empty:
    print("No signals found for today.")
else:
    print(f"Found {len(today_signals)} signals for today:")
    print(today_signals.head()) # Print head for brevity

## Combine with existing system
Market Regime Model
        ↓
Institutional Accumulation Detector
        ↓
Superperformance Detector
        ↓
Episodic Pivot Scanner
        ↓
AI Ranking Engine

What the system now does:
1. Market regime detection
2. Institutional accumulation detection
3. Episodic Pivot scanner
4. AI ranking engine
5. Superperfomance detector
6. Portfolio simulation
7. Research dashboard

In [ ]:

# ===============================
# FULL EP TRADING SYSTEM (FINAL)
# ===============================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from tqdm.auto import tqdm
import requests

from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# -------------------------------
# CONFIG
# -------------------------------
START_DATE = (datetime.today() - timedelta(days=365*5)).strftime('%Y-%m-%d')
END_DATE = datetime.today().strftime('%Y-%m-%d')


# -------------------------------
# DATA
# -------------------------------
def get_sp500_tickers():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors

    tables = pd.read_html(response.text)
    df = tables[0]
    tickers = df['Symbol'].str.replace('.', '-', regex=False).tolist()
    return tickers


def download_prices(tickers):
    data = yf.download(
        tickers,
        start=START_DATE,
        end=END_DATE,
        group_by='ticker',
        auto_adjust=True,
        progress=False
    )
    return data


def load_spy():
    spy = yf.download("SPY", start=START_DATE, end=END_DATE)
    spy.columns = spy.columns.droplevel(1)
    return spy


# -------------------------------
# MARKET REGIME
# -------------------------------
def market_features(spy):
    df = spy.copy()
    close = df["Close"]

    df["ret_3m"] = close.pct_change(63)
    df["ma50"] = close.rolling(50).mean()
    df["ma200"] = close.rolling(200).mean()

    df["trend"] = close / df["ma200"]

    return df


def label_regime(df):
    regime = []

    for i in range(len(df)):
        if df["Close"].iloc[i] > df["ma200"].iloc[i]:
            regime.append(1)
        else:
            regime.append(-1)

    df["regime"] = regime
    return df


# -------------------------------
# ACCUMULATION
# -------------------------------
def accumulation_score(price):
    close = price["Close"]
    volume = price["Volume"]

    vol_ratio = volume.iloc[-1] / volume.rolling(50).mean().iloc[-1]
    trend = close.iloc[-1] / close.rolling(60).mean().iloc[-1]

    return vol_ratio * 50 + trend * 50


# -------------------------------
# EP SCANNER
# -------------------------------
def scan_ep(price):

    signals = []

    for i in range(150, len(price)-10):

        trend = price["Close"].iloc[i] / price["Close"].iloc[i-126] - 1
        if trend < 0.2:
            continue

        base = price.iloc[i-40:i-5]
        base_range = (base["High"].max() - base["Low"].min()) / base["High"].max()

        if base_range > 0.35:
            continue

        gap = (price["Open"].iloc[i] - price["Close"].iloc[i-1]) / price["Close"].iloc[i-1]
        vol = price["Volume"].iloc[i] / price["Volume"].iloc[i-20:i].mean()

        if abs(gap) > 0.05 and vol > 1.8:
            signals.append({
                "Date": price.index[i],
                "Entry": price["Close"].iloc[i],
                "Stop": price["Low"].iloc[i],
                "Index": i
            })

    return signals


# -------------------------------
# FEATURES
# -------------------------------
def extract_features(price):

    close = price["Close"]
    volume = price["Volume"]

    return [
        close.pct_change(21).iloc[-1],
        close.pct_change(63).iloc[-1],
        close.pct_change(126).iloc[-1],
        volume.iloc[-1] / volume.rolling(20).mean().iloc[-1],
        close.pct_change().std()
    ]


# -------------------------------
# BACKTEST
# -------------------------------
def run_trade(price, signal):

    i = signal["Index"]
    entry = price["Open"].iloc[i+1]
    stop = signal["Stop"]

    risk = entry - stop

    for j in range(i+1, min(i+60, len(price))):

        low = price["Low"].iloc[j]
        high = price["High"].iloc[j]

        if low <= stop:
            return price.index[j], stop, -1

        if high >= entry + 3*risk:
            return price.index[j], entry + 3*risk, 3

    exit_price = price["Close"].iloc[j]
    R = (exit_price - entry) / risk

    return price.index[j], exit_price, R


# -------------------------------
# MAIN
# -------------------------------
def run():

    print("Loading data...")
    tickers = get_sp500_tickers()
    data = download_prices(tickers)
    spy = load_spy()

    spy_feat = market_features(spy)
    spy_regime = label_regime(spy_feat)

    print("Scanning accumulation...")
    scores = []

    for t in tickers:
        try:
            price = data[t].dropna()
            if len(price) < 100:
                continue

            score = accumulation_score(price)
            scores.append((t, score))
        except:
            continue

    top = sorted(scores, key=lambda x: x[1], reverse=True)[:50]
    universe = [x[0] for x in top]

    print("Scanning EP signals...")
    trades = []

    for t in universe:
        price = data[t].dropna()
        signals = scan_ep(price)

        for s in signals:

            exit_date, exit_price, R = run_trade(price, s)

            trades.append({
                "Ticker": t,
                "Entry_Date": s["Date"],
                "Entry": s["Entry"],
                "Stop": s["Stop"],
                "Exit_Date": exit_date,
                "Exit": exit_price,
                "R": R
            })

    trades = pd.DataFrame(trades)

    trades["Profit"] = trades["Exit"] - trades["Entry"]

    print("\n===== RESULTS ===telek:")
    print("Trades:", len(trades))
    print("Avg R:", trades["R"].mean())
    print("Win Rate:", (trades["R"] > 0).mean())

    trades.to_csv("ep_trades.csv", index=False)
    print("Saved to ep_trades.csv")


if __name__ == "__main__":
    run()

In [ ]:
trades